## Downstream Analysis

Post analysis after zonation trajectory inference:
- Differentially expressed genes / metabolites (Feature assotiation tests)
- Expression imputation via thin-plane spline(TPS) / cubic interpolation?
- Gene-Metabolite cross-correlations
- Gender comparison, etc.

In [ ]:
import os
import gc
import sys

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import torch
import torch.nn as nn
import torch.nn.functional as F
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams['font.family'] = 'FreeSans'
rcParams.update({'font.size': 8})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, trajectory, configs, zonation, plot
import scFates as scf

Load data & latent representations:

In [ ]:
%ls -al ../results/

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'

sample_id = 'NIH_F5'
adata = IO.load_xenium(os.path.join(xenium_path, sample_id)) 
adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
adata, adata_desi = IO.filter_cells(adata, adata_desi)

# Load latent representations
latent = np.load('../results/lynx_6_NIH_F5.npy')
n_latent = latent.shape[1]

adata.obsm['X_z'] = latent
# sc.pp.normalize_total(adata)
# sc.pp.log1p(adata)
# sc.pp.neighbors(adata, use_rep='X_z')
# sc.tl.umap(adata)

# Latent representation assignmeng for DESI
adata_desi.obsm['X_z'] = latent
# adata_desi.obsm['X_umap'] = adata.obsm['X_umap']

### Trajectory inference:

---

In [ ]:
dist_metric = 'euclidean'
n_nodes=10

# Xenium
trajectory.compute_trajectory(
    adata, 
    use_rep='X_z',
    n_nodes=n_nodes,
    dist_metric=dist_metric, 
)

# DESI
trajectory.compute_trajectory(
    adata_desi, 
    use_rep='X_z',
    n_nodes=n_nodes,
    dist_metric=dist_metric,
)
                              
gc.collect()

In [ ]:
# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['zone'] = adata.obs['zone'].astype(int).max() - adata.obs['zone'].astype(int)
adata.obs['zone'] = adata.obs['zone'].astype('category')

adata_desi.obs['t'] = adata_desi.obs['t'].max() - adata_desi.obs['t']
adata_desi.obs['zone'] = adata_desi.obs['zone'].astype(int).max() - adata_desi.obs['zone'].astype(int)
adata_desi.obs['zone'] = adata_desi.obs['zone'].astype('category')

In [ ]:
scf.pl.graph(adata, basis="umap")

In [ ]:
plot.disp_trajectory(adata, title='Spatial Gradients\n (LYNX)')

sq.pl.spatial_scatter(
    adata, color='t', 
    title='Spatial Gradients\n (LYNX)',
    cmap='RdBu_r', size=20, img=False
)

### Test Associations (fitted expressions along the trajectory)

In [ ]:
# PV-tip as root
scf.tl.test_association(adata, n_jobs=20)
scf.tl.test_association(adata, reapply_filters=True, A_cut=0.1)
scf.tl.root(adata, n_nodes+1) 
scf.tl.fit(adata, n_jobs=20)

scf.tl.test_association(adata_desi, n_jobs=20)
scf.tl.test_association(adata_desi, reapply_filters=True, A_cut=0.1)
scf.tl.root(adata_desi, 7)
scf.tl.fit(adata_desi, n_jobs=20)

gc.collect()

**Notes**<br>Significant DEGs: directly subsetting adata<br>
Fitted DEG expressions: `adata.layers['fitted']`

### Cell-type dynamics

- (1). Cell type compositions & (2) co-localization dynamics along the trajectory

In [ ]:
# Load cell-type annotation
annots = IO.load_xenium(os.path.join(xenium_path, sample_id), raw_count=False).obs['cell_type']
annots = annots.loc[adata.obs_names]

dynamics_df = utils.get_celltype_dynamics(adata, annots)
plot.disp_celltype_dynamics(
    dynamics_df, 
    ncol=4,
    # savedir='../results/plots/cell_dynamics.pdf'
)

dynamics_df.to_csv('../results/plots/dynamics_{}.csv'.format(sample_id), index=True)

### Xenium - DESI correlations

In [ ]:
fitted_xenium = adata.layers['fitted']  # [N x G]
fitted_desi = adata_desi.layers['fitted']  # [N x M]
fitted_concat = np.hstack((fitted_xenium, fitted_desi))
fitted_concat.shape

In [ ]:
gene_corr = np.corrcoef(fitted_xenium.T)
gene_corr_df = pd.DataFrame(
    gene_corr,
    index=list(adata.var_names),
    columns=list(adata.var_names)
)

metabolite_corr = np.corrcoef(fitted_desi.T)
metabolite_corr_df = pd.DataFrame(
    metabolite_corr,
    index=list(adata_desi.var_names),
    columns=list(adata_desi.var_names)
)
    
cross_corr = np.corrcoef(fitted_concat.T)
cross_corr_df = pd.DataFrame(
    cross_corr[:adata.shape[1], adata.shape[1]:], 
    index=list(adata.var_names),
    columns=list(adata_desi.var_names)
)

In [ ]:
# Save tentative heatmaps
gene_corr_df.to_csv('../results/xenium_corr_{}.csv'.format(sample_id), index=True)
metabolite_corr_df.to_csv('../results/DESI_corr_{}.csv'.format(sample_id), index=True)
cross_corr_df.to_csv('../results/xenium_DESI_corr_{}.csv'.format(sample_id), index=True)

In [ ]:
### Extract the clustered data
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots

g = sns.clustermap(metabolite_corr_df, cmap='RdBu_r')
clustered_data = g.data2d

### Create a dendrogram for the rows
dendro_row = ff.create_dendrogram(clustered_data.values, orientation='left', labels=clustered_data.index)
for trace in dendro_row['data']:
    trace['yaxis'] = 'y2'  # Linking this dendrogram to a different y-axis

### Create a dendrogram for the columns
dendro_col = ff.create_dendrogram(clustered_data.values.T, orientation='bottom', labels=clustered_data.columns)
for trace in dendro_col['data']:
    trace['xaxis'] = 'x2'  # Linking this dendrogram to a different x-axis

### Create main heatmap
heatmap = go.Heatmap(
    z=clustered_data.values,
    x=clustered_data.columns,
    y=clustered_data.index,
    colorscale='RdBu_r'
)

In [ ]:
### Combine Dendrograms and Heatmap into a Single Plotly Figure
fig = make_subplots(
    rows=2, cols=2,
    row_heights=[0.2, 0.8], column_widths=[0.8, 0.2],
    specs=[[{"type": "xy"}, None],
           [{"type": "heatmap"}, {"type": "xy"}]],
    horizontal_spacing=1, vertical_spacing=0.1
)

# Add the column dendrogram (top side)
for trace in dendro_col['data']:
    fig.add_trace(trace, row=1, col=1)

# Add the row dendrogram (right side)
for trace in dendro_row['data']:
    fig.add_trace(trace, row=2, col=2)

# Add the heatmap (bottom-left corner)
fig.add_trace(heatmap, row=2, col=1)

### Layout
# Update specific axes settings
fig.update_xaxes(domain=[0, 0.8], row=2, col=1)  # Heatmap X-axis
fig.update_yaxes(domain=[0.04, 0.76], row=2, col=1)  # Heatmap Y-axis

# Column dendrogram
fig.update_xaxes(domain=[0, 0.8], showticklabels=False, row=1, col=1)  
fig.update_yaxes(domain=[0.78, 1], showticklabels=False, row=1, col=1)  

# Row dendrogram
fig.update_xaxes(domain=[0.82, 1], showticklabels=False, row=2, col=2) 
fig.update_yaxes(domain=[0, 0.8], showticklabels=False, row=2, col=2) 

### Final layout adjustments
fig.update_layout(
    height=800, width=800,
    showlegend=False,
    hovermode='closest',
    plot_bgcolor='white',
    margin=dict(l=100, r=50, t=50, b=100),
)


In [ ]:
# fig.write_html("../results/Xenium_heatmap_{}.html".format(sample_id))
fig.write_html("../results/DESI_heatmap_{}.html".format(sample_id))
# fig.write_html("../results/Xenium_DESI_heatmap_{}.html".format(sample_id))

gc.collect()

### All-samples

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/desi/'
indir = '../results/'
outdir = '../results/downstream/'

sample_ids = sorted([
    sample_id for sample_id in os.listdir(xenium_path)
    if os.path.isdir(os.path.join(xenium_path, sample_id))
])


In [ ]:
from importlib import reload
reload(plot)
reload(utils)

#### Trajectory, Dynamics & Interpolation

In [ ]:
n_latent = 6
n_nodes = 10  # number of nodes for principal graph constrcution
n_zones = 5
n_bins = 500
n_desi_features = 100

cutoff_thresh = 0.1  # significance level for Test association 
dist_metric = 'euclidean'

outdir = '../results/downstream/'
if not os.path.isdir(outdir):
    os.makedirs(outdir, exist_ok=True)


# TMP: inversed 't': NIH_F4

for sample_id in sample_ids:
    print('Downstream analysis for {}'.format(sample_id))
    
    #-------------
    # Load data
    #-------------
    # latent = np.load('../results/lynx_{0}_{1}.npy'.format(n_latent, sample_id))
    latent = np.load('../results/lynx_{0}_{1}_new.npy'.format(n_latent, sample_id))

    adata = IO.load_multiomics(
        sample_id, xenium_path, desi_path, 
        n_features=None,
        project=True 
    )

    # adata_desi = sc.AnnData(adata.obsm['X_aux'])
    # adata_desi.obs_names = adata.obs_names
    # adata_desi.var_names = adata.uns['aux_features']

    adata.obsm['X_z'] = latent
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    sc.pp.neighbors(adata, use_rep='X_z')
    sc.tl.umap(adata)

    # adata_desi.obsm['X_z'] = latent
    # adata_desi.obsm['X_umap'] = adata.obsm['X_umap']

    #---------------------
    # Compute trajectory
    #---------------------
    trajectory.compute_trajectory(
        adata, 
        use_rep='X_z',
        n_nodes=n_nodes,
        dist_metric=dist_metric,
    )

    # trajectory.compute_trajectory(
    #     adata_desi,
    #     use_rep='X_z',
    #     n_nodes=n_nodes,
    #     dist_metric=dist_metric,
    # )

    # Optional: rotate trajectory direction by 180-deg
    if sample_id != 'NIH_F4':
        adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
    # adata_desi.obs['t'] = adata_desi.obs['t'].max() - adata_desi.obs['t']

    # tmp: checking quality
    # plot.disp_trajectory(adata, title='Spatial Gradients ({})\n (LYNX)'.format(sample_id))
    # sq.pl.spatial_scatter(
    #     adata, color='t', 
    #     title='Spatial Gradients ({})\n (LYNX)'.format(sample_id),
    #     cmap='RdBu_r', size=20, img=False
    # )
    # utils.get_zonations(adata, n_zones=n_zones, show=True)
    # sq.pl.spatial_scatter(
    #     adata, color='zone', 
    #     cmap='RdBu_r', size=20, img=False, 
    #     title='Zonation\n'+'LYNX'
    # )
    # plt.show()

    #---------------------------------------------------------------
    # Statistical tests & fitting expressions along the trajectory
    #---------------------------------------------------------------
    scf.tl.test_association(adata, n_jobs=20)
    scf.tl.test_association(adata, reapply_filters=True, A_cut=0.1)
    scf.tl.root(adata, n_nodes+1)  # "PV"-tip as root
    scf.tl.fit(adata,n_jobs=20)

    scf.tl.test_association(adata_desi, n_jobs=20)
    scf.tl.test_association(adata_desi, reapply_filters=True, A_cut=0.1)
    scf.tl.root(adata_desi, n_nodes+1)
    scf.tl.fit(adata_desi, n_jobs=20)

    # Extract fitted features sorted along trajectory
    fitted_gexp_df = utils.sort_fitted_expr(adata)    
    binned_gexp_df = plot.disp_fitted_expr(
        fitted_gexp_df, n_bins=n_bins, return_expr=True,
        savedir=os.path.join(outdir, 'gene_interp_{}'.format(sample_id))
    )

    fitted_gexp_df.to_csv(os.path.join(
        outdir,
        'fitted_genes.csv_{}'.format(sample_id)
    ), index=True)
    # binned_gexp_df.to_csv(os.path.join(outdir, 'gene_interp_{}.csv'.format(sample_id)), index=True)

    fitted_mexp_df = utils.sort_fitted_expr(adata_desi)    
    binned_mexp_df = plot.disp_fitted_expr(
        fitted_mexp_df, n_bins=n_bins, return_expr=True,
        savedir=os.path.join(outdir, 'metabolite_interp_{}'.format(sample_id))
    )
    
    fitted_mexp_df.to_csv(os.path.join(
        outdir,
        'fitted_metabolites_{}.csv'.format(sample_id)
    ), index=True)
    # binned_mexp_df.to_csv(os.path.join(outdir, 'metabolite_interp_{}.csv)'.format(sample_id)), index=True)

    del fitted_gexp_df, fitted_mexp_df, binned_gexp_df, binned_mexp_df
    gc.collect()

    #-----------------------------
    # Compute cell-type dynamics
    #-----------------------------
    # annots = IO.load_xenium(os.path.join(xenium_path, sample_id), raw_count=False).obs['cell_type']
    # annots = annots.loc[adata.obs_names]

    # dynamics_df = utils.get_celltype_dynamics(adata, annots)
    # plot.disp_dynamics(
    #     dynamics_df, ncols=4, 
    #     savedir=os.path.join(outdir, 'dynamics_{}.pdf'.format(sample_id))
    # )

    # dynamics_df.to_csv(os.path.join(outdir, 'dynamics_{}.csv'.format(sample_id)))

    # del dynamics_df
    # del adata, adata_desi
    del adata
    gc.collect()

    print('===================\n\n')


#### Enrichment analysis

Find the optimal **segmented (piecewise) regression** w/ (n-1) breakpoints
that fit smoothed trajectory $\gamma$ the best

Zonnation-specific differential abundance features, save output files for pathway analysis

3-zones:

In [ ]:
n_latent = 6
n_nodes = 10  # number of nodes for principal graph constrcution
n_zones = 3
n_bins = 500
n_desi = 100

cutoff_thresh = 0.1  # significance level for Test association 
dist_metric = 'knn'

outdir = '../results/downstream/'
if not os.path.isdir(outdir):
    os.makedirs(outdir, exist_ok=True)

adatas = []

pathway_dir = os.path.join(outdir, 'pathways_'+str(n_zones))
if not os.path.isdir(pathway_dir):
    os.makedirs(pathway_dir, exist_ok=True)

for sample_id in sample_ids:
    # Load data & latent manifold
    print('Computing differentially expressed features for {}...'.format(sample_id))
    adata = IO.load_multiomics(
        sample_id, xenium_path, desi_path, 
        n_features=n_desi,
        project=True
    )
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)

    adata_desi = sc.AnnData(adata.obsm['X_aux'])
    adata_desi.var_names = adata.uns['aux_features']
    adata_desi.obs_names = adata.obs_names.copy()
    adata_desi.obsm['spatial'] = adata.obsm['spatial'].copy()
    adata_desi.uns['spatial'] = adata.obsm['spatial'].copy()
 
    # adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
    # adata, adata_desi = IO.filter_cells(adata, adata_desi)

    latent = np.load('../results/lynx_{0}_{1}.npy'.format(n_latent, sample_id))
    adata.obsm['X_z'] = latent
    adata_desi.obsm['X_z'] = latent

    # Load interp features, filter ubiquitous features
    # fitted_gexp_df = pd.read_csv(os.path.join(outdir, '{}_fitted_genes.csv'.format(sample_id)), index_col=[0])
    # adata = adata[:, fitted_gexp_df.columns]

    # fitted_metabolite_df = pd.read_csv(os.path.join(outdir, '{}_fitted_metabolites.csv'.format(sample_id)), index_col=[0])
    # adata_desi = adata_desi[:, fitted_metabolite_df.columns]

    # Compute trajectory (save next time!)
    print('\n\n\tComputing trajectory...')
    trajectory.compute_trajectory(
        adata, 
        use_rep='X_z',
        n_nodes=n_nodes,
    )

    trajectory.compute_trajectory(
        adata_desi,
        use_rep='X_z',
        n_nodes=n_nodes,
    )

    # Optional: rotate trajectory direction by 180-deg
    adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
    adata_desi.obs['t'] = adata_desi.obs['t'].max() - adata_desi.obs['t']

    # del fitted_gexp_df, fitted_metabolite_df
    gc.collect()

    # Compute zone-wise DEGs / DEMs
    print('\n\n\tComputing DEGs/DEMs...')
    utils.get_zonation_features(
        adata, adata_desi, 
        n_zones=n_zones, n_bins=n_bins, sample_id=sample_id, 
        show=True
    )

    adatas.append(adata)

    zone_labels = np.unique(adata.obs['zone'])
    for label in zone_labels:
        adata.uns['zones'][str(label)].to_csv(
            os.path.join(pathway_dir, '{0}_zone_{1}_genes.csv'.format(sample_id, label)), 
            sep='\t', index=False
        )
        adata_desi.uns['zones'][str(label)].to_csv(os.path.join(
            pathway_dir, '{0}_zone_{1}_metabolites.csv'.format(sample_id, label)),
            sep='\t', index=False
        )

    print('==========================\n\n')
    del adata, adata_desi # zone_labels
    gc.collect()


5-zones:

In [ ]:
n_latent = 6
n_nodes = 10  # number of nodes for principal graph constrcution
n_zones = 5
n_bins = 1000
n_desi = 100

cutoff_thresh = 0.1  # significance level for Test association 
dist_metric = 'knn'

outdir = '../results/downstream/'
if not os.path.isdir(outdir):
    os.makedirs(outdir, exist_ok=True)

adatas = []

pathway_dir = os.path.join(outdir, 'pathways_'+str(n_zones))
if not os.path.isdir(pathway_dir):
    os.makedirs(pathway_dir, exist_ok=True)

for sample_id in sample_ids:
    # Load data & latent manifold
    print('Computing differentially expressed features for {}...'.format(sample_id))
    adata = IO.load_multiomics(
        sample_id, xenium_path, desi_path, 
        n_features=n_desi,
        project=True
    )
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)

    adata_desi = sc.AnnData(adata.obsm['X_aux'])
    adata_desi.var_names = adata.uns['aux_features']
    adata_desi.obs_names = adata.obs_names.copy()
    adata_desi.obsm['spatial'] = adata.obsm['spatial'].copy()
    adata_desi.uns['spatial'] = adata.obsm['spatial'].copy()
 
    # adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
    # adata, adata_desi = IO.filter_cells(adata, adata_desi)

    latent = np.load('../results/lynx_{0}_{1}.npy'.format(n_latent, sample_id))
    adata.obsm['X_z'] = latent
    adata_desi.obsm['X_z'] = latent

    # Load interp features, filter ubiquitous features
    # fitted_gexp_df = pd.read_csv(os.path.join(outdir, '{}_fitted_genes.csv'.format(sample_id)), index_col=[0])
    # adata = adata[:, fitted_gexp_df.columns]

    # fitted_metabolite_df = pd.read_csv(os.path.join(outdir, '{}_fitted_metabolites.csv'.format(sample_id)), index_col=[0])
    # adata_desi = adata_desi[:, fitted_metabolite_df.columns]

    # Compute trajectory (save next time!)
    print('\n\n\tComputing trajectory...')
    trajectory.compute_trajectory(
        adata, 
        use_rep='X_z',
        n_nodes=n_nodes,
    )

    trajectory.compute_trajectory(
        adata_desi,
        use_rep='X_z',
        n_nodes=n_nodes,
    )

    # Optional: rotate trajectory direction by 180-deg
    adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
    adata_desi.obs['t'] = adata_desi.obs['t'].max() - adata_desi.obs['t']

    # del fitted_gexp_df, fitted_metabolite_df
    gc.collect()

    # Compute zone-wise DEGs / DEMs
    print('\n\n\tComputing DEGs/DEMs...')
    utils.get_zonation_features(
        adata, adata_desi, 
        n_zones=n_zones, 
        # n_bins=n_bins, 
        sample_id=sample_id, 
        show=True
    )

    adatas.append(adata)

    zone_labels = np.unique(adata.obs['zone'])
    for label in zone_labels:
        adata.uns['zones'][str(label)].to_csv(
            os.path.join(pathway_dir, '{0}_zone_{1}_genes.csv'.format(sample_id, label)), 
            sep='\t', index=False
        )
        adata_desi.uns['zones'][str(label)].to_csv(os.path.join(
            pathway_dir, '{0}_zone_{1}_metabolites.csv'.format(sample_id, label)),
            sep='\t', index=False
        )

    print('==========================\n\n')
    del adata, adata_desi # zone_labels
    gc.collect()


In [ ]:
def annotate_peaks(peaks: pd.Series, ref: pd.DataFrame):
    
    def is_digit(str):
        return str.lstrip('-').replace('.', '').isdigit()

    peaks = peaks.apply(
        lambda x: 
        x.split('m/z')[0].strip(' ') if 'm/z' in x else x
    )

    annotated_peaks = peaks.apply(
        lambda x: ref[float(x)] 
        if (is_digit(x) and float(x) in ref.keys() and ref[float(x)] is not np.nan)
        else x
    )

    return annotated_peaks


# # Update annotations to differential abundance results

# reference = pd.read_csv(os.path.join(indir, 'metabolite_annotation.csv'), index_col=[0])
# reference = reference.to_dict()['Name']

# subdirs = ['pathways_3', 'pathways_5']
# col = 'm.z'

# for subdir in subdirs:
#     for smaple_id in sample_ids:

#         for file in sorted(os.listdir(os.path.join(outdir, subdir))):
#             filename = os.path.join(outdir, subdir, file)
#             if not os.path.isfile(filename) or 'metabolite' not in filename:
#                 continue

#             df = pd.read_csv(filename, sep='\t')
#             peaks = df[col].copy()
#             annotated_peaks = annotate_peaks(peaks, reference)
#             df.loc[:, col] = annotated_peaks

#     df.to_csv(filename, sep='\t', index=False)
    

#### Sex-specific gene expression along dynamics


In [ ]:
latent = np.load('../results/lynx_{0}_{1}_new.npy'.format(n_latent, sample_id))

adata = IO.load_multiomics(
    sample_id, xenium_path, desi_path, 
    n_features=None,
    project=True 
)
adata.obsm['X_z'] = latent

trajectory.compute_trajectory(
    adata, 
    use_rep='X_z',
    n_nodes=n_nodes,
)

adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']  # Convert PV-end as leftmost axis end

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
sample_gene = 'CYP1A1'

indices = np.argsort(adata.obs['t']).values
sample_binned_gene = utils.get_binned_expr(
    adata[:, sample_gene].to_df().iloc[indices].T,
    n_bins
)

GSEA Enrichr

In [ ]:
# GSEA on gene sets first
import networkx as nx
import gseapy as gp

In [ ]:
def get_enrichr(deg_df, title=None, n_retry=5, show_plot=False):
    """
    GSEA Enrichr analysis per zone
    """
    degs_up = np.unique(deg_df.loc[deg_df['logFC'] > 0, :]['genes'].tolist())
    degs_dw = np.unique(deg_df.loc[deg_df['logFC'] < 0, :]['genes'].tolist())

    count = 0
    for _ in range(n_retry):
        try:
            enr_up = gp.enrichr(
                degs_up,
                gene_sets='GO_Biological_Process_2021',
                outdir=None
            )
            enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    
            enr_dw = gp.enrichr(
                degs_dw,
                gene_sets='GO_Biological_Process_2021',
                outdir=None
            )
            enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
            break

        except Exception:
            count += 1
            continue

    if count == n_retry:
        #raise Warning('Failed on {}'.format(title))
        print('WARNING: Enrichr failed on {}'.format(title))
        return None

    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(5), enr_dw.res2d.head(5)])

    if show_plot:
        # # Combined plot
        gp.dotplot(
            enr_res,
            figsize=(3,5),
            x='UP_DW',
            x_order = ["UP","DOWN"],
            cmap = 'Reds',
            size=5,
            show_ring=True,
            title='GO_BP \n({})'.format(title)
        )


        # gp.barplot(
        #     pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()]), figsize=(3,5),
        #     group ='UP_DW',
        #     title ="GO_BP \n({})".format(title),
        #     color = ['b','r']
        # )

        # Network plot
        nodes, edges = gp.enrichment_map(enr_up.res2d)

        # build graph
        G = nx.from_pandas_edgelist(
            edges, source='src_idx', target='targ_idx',
            edge_attr=['jaccard_coef', 'overlap_coef', 'overlap_genes']
        )

        # Add missing node if there is any
        for node in nodes.index:
            if node not in G.nodes():
                G.add_node(node)

        fig, ax = plt.subplots(figsize=(8, 8))
        pos=nx.layout.spiral_layout(G)
        nx.draw_networkx_nodes(
            G,
            pos=pos,
            cmap=plt.cm.RdYlBu,
            node_size=list(nodes.Hits_ratio *1000)
        )
        nx.draw_networkx_labels(
            G,
            pos=pos,
            labels=nodes.Term.to_dict()
        )
        edge_weight = nx.get_edge_attributes(G, 'jaccard_coef').values()
        nx.draw_networkx_edges(
            G,
            pos=pos,
            width=list(map(lambda x: x*10, edge_weight)),
            edge_color='#CDDBD4'
        )
        plt.title('Enrichr Networks \n({})'.format(title), fontsize=15)
        plt.show()

    return enr_res



In [ ]:
n_zones = 3
subdir = os.path.join(outdir, 'pathways_{}'.format(n_zones))
zone_labels = ['zone '+str(i) for i in range(n_zones)]

for sample_id in sample_ids:
    print('Enrichment analysis on {}...'.format(sample_id))
    deg_filenames = sorted([
        os.path.join(subdir, f)
        for f in os.listdir(subdir)
        if os.path.isfile(os.path.join(subdir, f)) and 'gene' in f and sample_id in f
    ])
    deg_dfs = [pd.read_csv(f, sep='\t', index_col=None) for f in deg_filenames]

    for label, deg_df in zip(zone_labels, deg_dfs):
        _ = get_enrichr(deg_df, title=sample_id + ' ' + label, show_plot=True)

    del deg_df, label
    print('\n\n')

In [ ]:
# Sample data: NIH_F5
n_zones = 5
subdir = os.path.join(outdir, 'pathways_{}'.format(n_zones))
zone_labels = ['zone '+str(i) for i in range(n_zones)]

for sample_id in sample_ids:
    print('Enrichment analysis on {}...'.format(sample_id))
    deg_filenames = sorted([
        os.path.join(subdir, f)
        for f in os.listdir(subdir)
        if os.path.isfile(os.path.join(subdir, f)) and 'gene' in f and sample_id in f
    ])
    deg_dfs = [pd.read_csv(f, sep='\t', index_col=None) for f in deg_filenames]

    for label, deg_df in zip(zone_labels, deg_dfs):
        _ = get_enrichr(deg_df, title=sample_id + ' ' + label, show_plot=True)

    del deg_df, label
    print('\n\n')

---